# CLIP 零样本分类实战教程

本教程深入讲解如何使用 CLIP 进行零样本图像分类，包括：
- 零样本分类原理
- Prompt Engineering 技巧
- 多模板集成策略
- 层级分类实现

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn.functional as F
from typing import List, Dict, Tuple
import numpy as np

from clip import CLIP, CLIPConfig, create_clip_model

## 1. 零样本分类原理

### 核心思想

零样本分类利用 CLIP 的图像-文本对齐能力，无需任何训练样本即可进行分类：

```
传统分类: 图像 → CNN → 固定类别 softmax
零样本分类: 图像 → CLIP → 与文本描述相似度 → 动态类别
```

### 数学公式

$$P(y=c|x) = \frac{\exp(\text{sim}(f_v(x), f_t(t_c)) / \tau)}{\sum_{c'} \exp(\text{sim}(f_v(x), f_t(t_{c'})) / \tau)}$$

In [ ]:
class ZeroShotClassifier:
    """
    CLIP 零样本分类器
    
    核心流程:
    1. 将类别名转换为文本描述
    2. 编码图像和所有类别文本
    3. 计算相似度并归一化为概率
    """
    
    def __init__(self, model: CLIP, class_names: List[str], templates: List[str] = None):
        self.model = model
        self.class_names = class_names
        self.templates = templates or ["a photo of a {}"]
        
        # 预计算类别文本特征
        self.class_features = self._compute_class_features()
    
    def _compute_class_features(self) -> torch.Tensor:
        """预计算所有类别的文本特征（多模板平均）"""
        all_features = []
        
        for class_name in self.class_names:
            # 对每个类别应用所有模板
            texts = [template.format(class_name) for template in self.templates]
            
            # 模拟文本编码（实际使用时需要 tokenizer）
            # 这里用随机 token 演示
            input_ids = torch.randint(0, 49408, (len(texts), 77))
            
            with torch.no_grad():
                text_features = self.model.encode_text(input_ids)
                text_features = F.normalize(text_features, dim=-1)
                # 多模板平均
                class_feature = text_features.mean(dim=0)
                class_feature = F.normalize(class_feature, dim=-1)
            
            all_features.append(class_feature)
        
        return torch.stack(all_features)  # [num_classes, embed_dim]
    
    def classify(self, images: torch.Tensor, top_k: int = 5) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        对图像进行零样本分类
        
        Returns:
            probs: 类别概率 [batch_size, num_classes]
            top_k_indices: Top-K 类别索引
        """
        with torch.no_grad():
            # 编码图像
            image_features = self.model.encode_image(images)
            image_features = F.normalize(image_features, dim=-1)
            
            # 计算相似度
            logit_scale = self.model.logit_scale.exp()
            logits = logit_scale * image_features @ self.class_features.T
            
            # 转换为概率
            probs = F.softmax(logits, dim=-1)
            
            # 获取 Top-K
            top_k_probs, top_k_indices = probs.topk(top_k, dim=-1)
        
        return probs, top_k_indices

# 测试
model = create_clip_model("small")
classes = ["cat", "dog", "bird", "car", "airplane"]
classifier = ZeroShotClassifier(model, classes)

# 模拟图像输入
images = torch.randn(2, 3, 224, 224)
probs, top_k = classifier.classify(images)
print(f"Probabilities shape: {probs.shape}")
print(f"Top-3 predictions: {top_k}")

## 2. Prompt Engineering 技巧

Prompt 的设计对零样本分类性能影响巨大。

In [ ]:
# 不同领域的 Prompt 模板
PROMPT_TEMPLATES = {
    "general": [
        "a photo of a {}",
        "a photograph of a {}",
        "an image of a {}",
        "a picture of a {}",
    ],
    "imagenet": [
        "a bad photo of a {}",
        "a photo of many {}",
        "a sculpture of a {}",
        "a photo of the hard to see {}",
        "a low resolution photo of the {}",
        "a rendering of a {}",
        "graffiti of a {}",
        "a drawing of a {}",
        "a photo of the large {}",
        "a photo of the small {}",
    ],
    "food": [
        "a photo of {}, a type of food",
        "a photo of a delicious {}",
        "a photo of {} on a plate",
    ],
    "animal": [
        "a photo of a {}, a type of animal",
        "a photo of a {} in the wild",
        "a photo of a {} in a zoo",
    ],
    "scene": [
        "a photo of a {}",
        "a photo of the {}",
        "a scenic view of {}",
    ],
}

def get_domain_templates(domain: str) -> List[str]:
    """获取特定领域的 Prompt 模板"""
    return PROMPT_TEMPLATES.get(domain, PROMPT_TEMPLATES["general"])

print("ImageNet 模板示例:")
for t in PROMPT_TEMPLATES["imagenet"][:5]:
    print(f"  - {t.format('dog')}")

## 3. 多模板集成策略

使用多个模板并平均可以显著提升分类准确率。

In [ ]:
class EnsembleZeroShotClassifier:
    """
    多模板集成零样本分类器
    
    策略:
    1. 特征级集成: 平均多模板的文本特征
    2. 概率级集成: 平均多模板的预测概率
    3. 加权集成: 根据模板质量加权
    """
    
    def __init__(self, model: CLIP, class_names: List[str], templates: List[str]):
        self.model = model
        self.class_names = class_names
        self.templates = templates
    
    def feature_ensemble(self, images: torch.Tensor) -> torch.Tensor:
        """特征级集成 - 平均文本特征后计算相似度"""
        with torch.no_grad():
            image_features = self.model.encode_image(images)
            image_features = F.normalize(image_features, dim=-1)
            
            all_class_features = []
            for class_name in self.class_names:
                template_features = []
                for template in self.templates:
                    text = template.format(class_name)
                    input_ids = torch.randint(0, 49408, (1, 77))
                    feat = self.model.encode_text(input_ids)
                    template_features.append(F.normalize(feat, dim=-1))
                
                # 平均模板特征
                avg_feat = torch.stack(template_features).mean(dim=0)
                avg_feat = F.normalize(avg_feat, dim=-1)
                all_class_features.append(avg_feat)
            
            class_features = torch.cat(all_class_features, dim=0)
            logits = self.model.logit_scale.exp() * image_features @ class_features.T
            
        return F.softmax(logits, dim=-1)
    
    def probability_ensemble(self, images: torch.Tensor) -> torch.Tensor:
        """概率级集成 - 平均每个模板的预测概率"""
        all_probs = []
        
        with torch.no_grad():
            image_features = self.model.encode_image(images)
            image_features = F.normalize(image_features, dim=-1)
            
            for template in self.templates:
                class_features = []
                for class_name in self.class_names:
                    text = template.format(class_name)
                    input_ids = torch.randint(0, 49408, (1, 77))
                    feat = self.model.encode_text(input_ids)
                    class_features.append(F.normalize(feat, dim=-1))
                
                class_features = torch.cat(class_features, dim=0)
                logits = self.model.logit_scale.exp() * image_features @ class_features.T
                probs = F.softmax(logits, dim=-1)
                all_probs.append(probs)
        
        # 平均概率
        return torch.stack(all_probs).mean(dim=0)

# 测试集成分类器
templates = PROMPT_TEMPLATES["general"]
ensemble_clf = EnsembleZeroShotClassifier(model, classes, templates)

probs_feat = ensemble_clf.feature_ensemble(images)
probs_prob = ensemble_clf.probability_ensemble(images)

print(f"Feature ensemble probs: {probs_feat.shape}")
print(f"Probability ensemble probs: {probs_prob.shape}")

## 4. 层级分类实现

对于大规模类别，可以使用层级分类策略提升效率和准确率。

In [ ]:
class HierarchicalClassifier:
    """
    层级零样本分类器
    
    示例层级结构:
    - 动物
      - 哺乳动物: 猫, 狗, 马
      - 鸟类: 鹦鹉, 老鹰, 企鹅
    - 交通工具
      - 陆地: 汽车, 卡车, 摩托车
      - 空中: 飞机, 直升机
    """
    
    def __init__(self, model: CLIP, hierarchy: Dict[str, Dict[str, List[str]]]):
        self.model = model
        self.hierarchy = hierarchy
        
        # 构建各级分类器
        self.coarse_classes = list(hierarchy.keys())
        self.fine_classes = {}
        for coarse, subcats in hierarchy.items():
            self.fine_classes[coarse] = {}
            for subcat, classes in subcats.items():
                self.fine_classes[coarse][subcat] = classes
    
    def classify(self, images: torch.Tensor) -> Dict:
        """层级分类"""
        results = {
            "coarse": None,
            "medium": None,
            "fine": None,
            "confidence": None
        }
        
        with torch.no_grad():
            image_features = self.model.encode_image(images)
            image_features = F.normalize(image_features, dim=-1)
            
            # 第一级: 粗粒度分类
            coarse_probs = self._classify_level(image_features, self.coarse_classes)
            coarse_pred = coarse_probs.argmax(dim=-1)
            results["coarse"] = [self.coarse_classes[i] for i in coarse_pred]
            
            # 第二级: 中粒度分类 (基于粗粒度结果)
            # 简化: 只处理第一个样本
            coarse_name = results["coarse"][0]
            medium_classes = list(self.fine_classes[coarse_name].keys())
            medium_probs = self._classify_level(image_features, medium_classes)
            medium_pred = medium_probs.argmax(dim=-1)
            results["medium"] = [medium_classes[i] for i in medium_pred]
            
            # 第三级: 细粒度分类
            medium_name = results["medium"][0]
            fine_classes = self.fine_classes[coarse_name][medium_name]
            fine_probs = self._classify_level(image_features, fine_classes)
            fine_pred = fine_probs.argmax(dim=-1)
            results["fine"] = [fine_classes[i] for i in fine_pred]
            results["confidence"] = fine_probs.max(dim=-1).values.tolist()
        
        return results
    
    def _classify_level(self, image_features: torch.Tensor, classes: List[str]) -> torch.Tensor:
        """单级分类"""
        class_features = []
        for class_name in classes:
            input_ids = torch.randint(0, 49408, (1, 77))
            feat = self.model.encode_text(input_ids)
            class_features.append(F.normalize(feat, dim=-1))
        
        class_features = torch.cat(class_features, dim=0)
        logits = self.model.logit_scale.exp() * image_features @ class_features.T
        return F.softmax(logits, dim=-1)

# 定义层级结构
hierarchy = {
    "animal": {
        "mammal": ["cat", "dog", "horse"],
        "bird": ["parrot", "eagle", "penguin"]
    },
    "vehicle": {
        "land": ["car", "truck", "motorcycle"],
        "air": ["airplane", "helicopter"]
    }
}

hier_clf = HierarchicalClassifier(model, hierarchy)
results = hier_clf.classify(images)
print(f"Hierarchical classification results:")
print(f"  Coarse: {results['coarse']}")
print(f"  Medium: {results['medium']}")
print(f"  Fine: {results['fine']}")

## 5. 性能优化技巧

In [ ]:
class OptimizedZeroShotClassifier:
    """
    优化的零样本分类器
    
    优化策略:
    1. 预计算并缓存类别特征
    2. 批量处理图像
    3. 使用半精度推理
    """
    
    def __init__(self, model: CLIP, class_names: List[str], use_fp16: bool = True):
        self.model = model
        self.class_names = class_names
        self.use_fp16 = use_fp16
        
        # 预计算类别特征
        self._precompute_class_features()
    
    def _precompute_class_features(self):
        """预计算并缓存类别特征"""
        features = []
        with torch.no_grad():
            for class_name in self.class_names:
                input_ids = torch.randint(0, 49408, (1, 77))
                feat = self.model.encode_text(input_ids)
                features.append(F.normalize(feat, dim=-1))
        
        self.class_features = torch.cat(features, dim=0)
        if self.use_fp16:
            self.class_features = self.class_features.half()
    
    @torch.no_grad()
    def classify_batch(self, images: torch.Tensor, batch_size: int = 32) -> torch.Tensor:
        """批量分类"""
        all_probs = []
        
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size]
            if self.use_fp16:
                batch = batch.half()
            
            image_features = self.model.encode_image(batch.float())
            image_features = F.normalize(image_features, dim=-1)
            
            if self.use_fp16:
                image_features = image_features.half()
            
            logits = self.model.logit_scale.exp() * image_features @ self.class_features.T
            probs = F.softmax(logits.float(), dim=-1)
            all_probs.append(probs)
        
        return torch.cat(all_probs, dim=0)

# 测试优化分类器
opt_clf = OptimizedZeroShotClassifier(model, classes, use_fp16=False)
large_batch = torch.randn(10, 3, 224, 224)
probs = opt_clf.classify_batch(large_batch, batch_size=4)
print(f"Batch classification: {probs.shape}")

## 总结

本教程介绍了 CLIP 零样本分类的核心技术：

1. **基础原理**: 利用图像-文本对齐进行开放词汇分类
2. **Prompt Engineering**: 设计合适的文本模板提升性能
3. **多模板集成**: 特征级和概率级集成策略
4. **层级分类**: 处理大规模类别的高效方法
5. **性能优化**: 预计算、批处理、半精度推理